# 04 加权 A*：有界次优（w-admissible）


本课按百科条目写：先定义，再界限定理，再从第 03 课的 A* **只改一行** 写出加权 A*。调度台在最后，可跳过。

---

## 1. 定义

**加权 A\***（Weighted A\*，Pohl 1970 前后；也叫 \(w\)-A\*、inflated A\*）把 A\* 的键改成

\[
f_w(n)=g(n)+w\,h(n),\qquad w\ge 1
\]

- 输入：图、起点、终点、可采纳启发 \(h\)、权重 \(w\ge 1\)。
- 输出：一条 \(s\leadsto t\) 路径，其代价 \(\mathrm{cost}\le w\cdot OPT\)（见第 3 节）。路径的 **真实代价仍是 \(g(t)\)**，不是 \(f_w(t)\)。

直观：\(w>1\) 时更听信「还剩多少」，堆会更早弹出靠近终点的点，扩展更少，路可能更绕。

Anytime / ARA\*：先用大 \(w\) 很快出解，再把 \(w\) 降到 1 改善。本课只实现 **固定 \(w\)**。

---

## 2. 和 A* / Dijkstra / 贪心的关系

| 算法 | 堆键 | 保证 |
|------|------|------|
| Dijkstra / UCS | \(g\) | 最优（\(w\ge 0\)） |
| A\* | \(g+h\) | \(h\) 可采纳 ⇒ 最优 |
| 加权 A\* | \(g+w h\) | \(h\) 可采纳 ⇒ \(\le w\cdot OPT\) |
| 贪心最佳优先 | \(h\) | 无 |

\(w=1\) 就是 A\*。\(w\to\infty\) 时 \(g\) 相对可忽略，行为接近贪心。

网上还有 \(f=g/w+h\) 或 \(f=g+(2w-1)h\)。界限陈述不同。本仓库与 Wikipedia 常见形式一致：`g + w*h`。

---

## 3. 不变量（w-admissible）

设 \(h\) 可采纳（从不高估剩余）。加权 A\* 弹出终点时：

\[
g(t)\le w\cdot OPT
\]

证明梗概（对照 03 课 A\* 最优证明）：最优路上第一个还在 open 的点 \(n\) 满足 \(g(n)=g^*(n)\)，且 \(h(n)\le h^*(n)\)，于是

\[
f_w(n)=g^*(n)+w h(n)\le w\big(g^*(n)+h^*(n)\big)=w\cdot OPT
\]

堆先弹 \(f_w\) 最小的。若返回的 \(g(t)>w\cdot OPT\)，则当时 \(f_w(t)\ge g(t)>w\cdot OPT\ge f_w(n)\)，应先扩展 \(n\) 而不是终点。矛盾。

**注意：**

- \(w<1\) 没有「比最优还短」这回事，证明直接坏掉（\(w h\) 可能比真剩余还小得过头，堆序乱）。
- 不一致但可采纳的 \(h\) 加上「关闭后永不重开」，界限仍常被引用，严格证明要更小心。本课网格曼哈顿是一致的。
- 返回值必须读 **g**，不要把 \(f_w\) 当路径长。

---

## 4. 伪代码（与下面 Python 只差一个 \(w\)）

```
function WeightedAStar(G, s, t, h, w):
    g[s] ← 0;  came_from[s] ← NIL
    push heap (w * h(s), s)
    closed ← ∅
    while heap 非空:
        (f, u) ← pop 最小 f
        if u ∈ closed: continue          # 惰性堆过期条目
        关闭 u
        if u = t: return Reconstruct(came_from, t)
        for each 边 (u, v, c):
            alt ← g[u] + c
            if alt < g[v]:
                g[v] ← alt
                came_from[v] ← u
                push heap (alt + w * h(v), v)
    return FAILURE
```

复杂度同 A\*：惰性堆 \(O(E\log E)\)。\(w\) 越大 **实际** 扩展越少，最坏仍可能扫全图（\(h=0\) 或墙把启发「骗」没了）。

---

## 5. 手算（同一点、三个 w）

四连通、步长 1，曼哈顿。某点 \(g=4\)，\(h=6\)：

| w | f = g+w h | 含义 |
|---|-----------|------|
| 1 | 10 | 普通 A\* |
| 1.5 | 13 | 略偏信启发 |
| 3 | 22 | 很偏信启发 |

另一个更靠近终点的点 \(g=8\)，\(h=1\)：

- w=1：f=9，比上面的 10 还小，A\* 会先扩这个「已经走远但离终点近」的点。
- w=3：f=11，而第一点 f=22，堆会 **更早** 扩第二点——哪怕它的 g 更大。

墙图 S=(0,0)、G=(7,7)、竖墙 x=3 且 y=0..6、缺口 (3,7)：\(h(S)=14\)。w 越大，搜索越贴着「看起来朝 G」的格子，可能早早钻进死胡同再绕回来，路径变长、扩展变少。


In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "lib").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))


## 6. 实现：从第 03 课的 A* 加参数 w

下面这段就是 03 课的 `astar`，只多一个 `weight`，松弛后的堆键写成 `tentative + weight * h(...)`。


In [2]:
import heapq
from lib.maps import grid_wall
from lib.pretty import draw_grid


def manhattan(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])


def reconstruct(came_from, goal):
    node = goal
    path = [node]
    while came_from[node] is not None:
        node = came_from[node]
        path.append(node)
    path.reverse()
    return path


def weighted_astar(graph, start, goal, weight=1.0, h=manhattan, verbose=False):
    # weight=1 必须退回普通 A*。不要把 weight 写进 g。
    heap = []
    counter = 0
    g_score = {start: 0.0}
    came_from = {start: None}
    heapq.heappush(heap, (weight * h(start, goal), counter, start))
    closed = set()
    expanded = 0
    while heap:
        _f, _, current = heapq.heappop(heap)
        if current in closed:
            continue
        closed.add(current)
        expanded += 1
        g_cur = g_score[current]
        if verbose and expanded <= 8:
            hn = h(current, goal)
            print(f"弹出 {current}  g={g_cur:.0f}  h={hn:.0f}  f={g_cur + weight * hn:.1f}")
        if current == goal:
            return reconstruct(came_from, goal), g_cur, expanded
        for nxt, cost in graph.neighbors(current):
            tentative = g_cur + cost
            if tentative < g_score.get(nxt, float("inf")):
                g_score[nxt] = tentative
                came_from[nxt] = current
                counter += 1
                # 相对 A* 的唯一改动：键是 g + w*h
                heapq.heappush(heap, (tentative + weight * h(nxt, goal), counter, nxt))
    return None, float("inf"), expanded


graph = grid_wall()
s, t = (0, 0), (7, 7)
print(draw_grid(graph.width, graph.height, obstacles=graph.obstacles, start=s, goal=t))
print("w=1 前 8 次弹出：")
p1, c1, e1 = weighted_astar(graph, s, t, weight=1.0, verbose=True)
print("路径代价(g)", c1, "扩展", e1)


S起点 G终点 #墙 .空 o待扩展 x已扩展 *当前 @路径
S..#....
...#....
...#....
...#....
...#....
...#....
...#....
.......G
w=1 前 8 次弹出：
弹出 (0, 0)  g=0  h=14  f=14.0
弹出 (1, 0)  g=1  h=13  f=14.0
弹出 (0, 1)  g=1  h=13  f=14.0
弹出 (2, 0)  g=2  h=12  f=14.0
弹出 (1, 1)  g=2  h=12  f=14.0
弹出 (0, 2)  g=2  h=12  f=14.0
弹出 (2, 1)  g=3  h=11  f=14.0
弹出 (1, 2)  g=3  h=11  f=14.0
路径代价(g) 14.0 扩展 29


## 7. 逐行（相对 A*）

| 位置 | 干什么 |
|------|--------|
| `weight=1` 默认 | 退回 A\*，便于对照 |
| 起点键 `weight * h(s)` | g(s)=0，f=w h |
| `tentative = g_cur + cost` | 真实代价，**不含** w |
| `push (tentative + weight * h(nxt))` | 唯一改动 |
| 返回 `g_cur` | 给评测、给「是否 ≤ w·OPT」 |

---

## 8. 实验：界限成立吗？扩展少了吗？


In [3]:
opt_path, opt, opt_e = weighted_astar(graph, s, t, weight=1.0)
print("OPT", opt, "扩展", opt_e)
for w in (1.0, 1.5, 3.0, 10.0):
    path, cost, n_exp = weighted_astar(graph, s, t, weight=w)
    bound = w * opt
    ok = cost <= bound + 1e-9
    print(f"w={w:4}  代价={cost:5}  上限={bound:6.1f}  扩展={n_exp:4}  界限={ok}  比OPT长={cost - opt:.0f}")


OPT 14.0 扩展 29
w= 1.0  代价= 14.0  上限=  14.0  扩展=  29  界限=True  比OPT长=0
w= 1.5  代价= 14.0  上限=  21.0  扩展=  15  界限=True  比OPT长=0
w= 3.0  代价= 14.0  上限=  42.0  扩展=  15  界限=True  比OPT长=0
w=10.0  代价= 14.0  上限= 140.0  扩展=  15  界限=True  比OPT长=0


你应看到：w 变大 → 扩展下降（这张墙图上很明显），代价 **不小于** OPT，且不超过 `w*OPT`。

「w 越大路径单调变长」**不成立**：堆序一变，可能碰巧走出和 OPT 一样的路，也可能某次中间 w 更绕。界限是上界，不是单调性。

---

## 9. 常见 bug

1. 把 \(f_w\) 当路径长打印：数字里含 h，看起来「比 Dijkstra 还长一截」，其实没走那么远。
2. `w<1`：证明作废，还可能比「可采纳 A\*」扩得更怪。
3. 以为加权 A\* 路径更短：通常 **更长、更快**。
4. 用不可采纳 h 再乘 w：连 \(w\cdot OPT\) 都不保证。
5. 和 anytime 搞混：固定 w 跑一次就结束，不会自动降 w。

**库对照：** `lib.search.astar(..., weight=w)` 与上面同一行。`lib.algos.single_agent.weighted_astar` 只是它的别名。


In [4]:
from lib.search import astar as astar_lib

r = astar_lib(graph, s, t, weight=1.5, record=False)
print("库 weight=1.5 代价", r.cost, "扩展", r.expanded, "extra", r.extra)


库 weight=1.5 代价 14.0 扩展 15 extra {'weight': 1.5}


In [5]:
from lib.studio.widget import PlannerStudio
from lib.search import astar as astar_lib

r = astar_lib(graph, s, t, weight=1.5)
studio = PlannerStudio()
studio.show_search({**graph.to_studio(), "start": [0, 0], "goal": [7, 7]}, r, "weighted A*")
studio


单步：波前比 w=1 更尖、更朝 G。墙右侧仍要等绕过缺口。

## 条目小结

| 项目 | 内容 |
|------|------|
| 公式 | \(f=g+w h\)，\(w\ge 1\) |
| 保证 | h 可采纳 ⇒ 代价 \(\le w\cdot OPT\) |
| 实现差 | 相对 A\* 只改堆键 |
| 退化 | w=1 → A\*；w 很大 → 接近贪心 |
| 下一课 | 图中途变了 → D\* Lite，不必每次重跑 A\* |

## 练习

1. 证明或举反例：w 越大，返回路径的 g **单调**变长。
2. 连续两次：先 w=3 再 w=1。第二次不应比 OPT 差。自己跑。
3. 把 `return` 改成返回 `_f` 而不是 `g_cur`，w=3 时数字会怎样？为什么评测会误判？
